# HeartlyPE — CV reproducible en Colab (GPU)

**Antes de empezar:** `Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)`.

Corre las celdas en orden. Los resultados se guardan en **Google Drive** (`MyDrive/heartlype_runs/`), así que si Colab se desconecta basta con volver a correr todo: los jobs (semilla × fold) ya terminados se saltan.

## 1. Clonar el código y montar Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!rm -rf /content/ecg-arrhythmia-thesis
!git clone -q -b data-artifacts https://github.com/ClaudiaSifuentes/ecg-arrhythmia-thesis.git /content/ecg-arrhythmia-thesis
%cd /content/ecg-arrhythmia-thesis
!git log --oneline -1

## 2. Dependencias

In [ ]:
!pip -q install wfdb neurokit2 pyyaml scikit-learn
import torch; print(torch.__version__, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU (revisa el tipo de entorno)')

## 3. Datos MIT-BIH + INCART (desde la rama `data-incart`, vía Git LFS)

In [ ]:
# Colab no trae Git LFS: sin esto los .npy bajan como "punteros" de ~130 bytes
!apt-get -qq install -y git-lfs > /dev/null && git lfs install --skip-repo
!rm -rf /content/data_incart
!git clone -q --depth 1 -b data-incart https://github.com/ClaudiaSifuentes/ecg-arrhythmia-thesis.git /content/data_incart
!mkdir -p data/processed/mitbih_incart && cp /content/data_incart/data/processed/mitbih_incart/*.npy data/processed/mitbih_incart/
!ls -la data/processed/mitbih_incart/
import os
size = os.path.getsize('data/processed/mitbih_incart/X_beats.npy')
assert size > 200_000_000, f'X_beats.npy pesa {size} bytes: LFS no descargó los datos. Vuelve a correr esta celda.'
print('OK: datos reales descargados')

## 4. Salidas persistentes en Drive
Los resultados (`reports/cv`, `reports/oof`, `reports/thresholds`, `reports/test`, `models/cv`, `models/final`) se enlazan a Drive.

In [ ]:
import os
RUNS = '/content/drive/MyDrive/heartlype_runs'
for d in ['reports/cv', 'reports/oof', 'reports/thresholds', 'reports/test', 'models/cv', 'models/final']:
    target = f'{RUNS}/{d}'
    os.makedirs(target, exist_ok=True)
    if os.path.isdir(d) and not os.path.islink(d):
        !cp -rn {d}/. {target}/ 2>/dev/null; rm -rf {d}
    if not os.path.exists(d):
        os.makedirs(os.path.dirname(d), exist_ok=True); os.symlink(target, d)
!ls -la reports models

## 5. Mapa registro → paciente de INCART
Ya viene en el repo (`configs/incart_subjects.json`, 75 registros → 32 pacientes, leído del campo `patient N` de los encabezados). Esta celda solo lo verifica.

In [ ]:
import json
m = json.load(open('configs/incart_subjects.json'))['subject_of_record']
print(len(m), 'registros INCART ->', len(set(m.values())), 'pacientes')   # debe decir 75 -> 32

## 6. Tests de determinismo (ahora en GPU)

In [ ]:
!PYTHONPATH=. python -m pytest -q tests/test_protocol.py

## 7. Baseline oficial E12 (3 semillas × 5 folds)
Receta E12 sin cambios; early stopping con validación interna por paciente; umbral SVEB 0.24 del config aplicado igual en CV y test.

In [ ]:
!PYTHONPATH=. python scripts/run_cv.py --config configs/experiments/E12.yaml

## 8. Barrido de umbrales sobre OOF y variantes post-hoc (ensamble / temperatura)

In [ ]:
!PYTHONPATH=. python scripts/threshold_sweep_oof.py --exp E12
!PYTHONPATH=. python scripts/posthoc_variants.py --exp E12

## 9. Experimento E13: contexto RR (un solo cambio)

In [ ]:
!PYTHONPATH=. python scripts/run_cv.py --config configs/experiments/E13_rrctx.yaml
!PYTHONPATH=. python scripts/threshold_sweep_oof.py --exp E13_rrctx

## 10. Devolver los resultados a GitHub (rama `colab-results`)
Necesitas un **token de GitHub** (es como una contraseña temporal solo para este repo):
1. Entra a https://github.com/settings/personal-access-tokens/new
2. *Token name*: `colab`; *Expiration*: 7 días.
3. *Repository access* → **Only select repositories** → `ecg-arrhythmia-thesis`.
4. *Permissions* → *Repository permissions* → **Contents: Read and write**.
5. **Generate token**, cópialo (empieza con `github_pat_`) y pégalo cuando esta celda lo pida.

No lo pegues en el chat ni en el código: la celda lo pide de forma oculta y no queda guardado. Cuando terminemos puedes borrarlo en esa misma página.

In [ ]:
from getpass import getpass
token = getpass('GitHub token: ')
!git config user.email "csifuentes398@gmail.com" && git config user.name "ClaudiaSifuentes"
!git checkout -q -B colab-results
!mkdir -p results_colab && cp -rL reports/cv reports/oof reports/thresholds reports/test results_colab/ 2>/dev/null; cp configs/incart_subjects.json results_colab/
!find results_colab -name 'part_*.npz' -delete
!git add -f results_colab configs/incart_subjects.json && git commit -qm "results: Colab CV runs" && git push -q -f https://{token}@github.com/ClaudiaSifuentes/ecg-arrhythmia-thesis.git colab-results
print('listo: rama colab-results')